# AI Research Workflow · 4일차

**결과 해석과 논문 작성** (2026-11-24)

- 위에서부터 차례로 실행합니다. 셀을 고른 뒤 Shift + Enter 를 누르면 됩니다.
- 예제 데이터는 교육용으로 만든 **가상 데이터**입니다. 실제 시험 결과가 아니므로 논문 · 보고서에 인용하지 않습니다.
- 이 노트북은 사본으로 열립니다. 고친 내용을 남기려면 「파일 → Drive에 사본 저장」을 누릅니다.
- 설명과 프롬프트는 학습 사이트 4일차 본문에 있습니다.

## 결과를 주장으로

### 논문용 그림으로 다듬기 (Colab)

그림의 숫자는 표(summary)와 같아야 합니다. 그림을 고친 뒤에는 표를 다시 출력해 대조하세요. 축 이름을 한글로 쓰려면 한글 글꼴 설정이 따로 필요합니다.

In [ ]:
# 3일차에 만든 정제본으로 그림 1을 다시 그립니다.
# 정제본 파일을 Colab 왼쪽 「파일」에 올린 뒤 실행하세요.
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("https://raw.githubusercontent.com/jungaistar/materials/main/grad/battery_cycling_clean.csv")

# 충방전율 1.0C 만 골라 온도 조건별로 비교합니다.
sub = df[df["c_rate"] == 1.0]

# 분석 단위는 셀입니다. 사이클 시점마다 "셀들의" 평균과 표준편차, 셀 수를 구합니다.
summary = (
    sub.groupby(["temp_c", "cycle"])["retention_pct"]
       .agg(mean="mean", sd="std", n="count")
       .reset_index()
)
print(summary.head(12))          # 그림에 들어갈 숫자를 먼저 눈으로 확인합니다.

# 색만으로 구분하지 않도록 표식 모양과 선 종류를 함께 바꿉니다.
style = {25: dict(marker="o", linestyle="-"), 45: dict(marker="^", linestyle="--")}

fig, ax = plt.subplots(figsize=(3.5, 2.8))     # 논문 한 단 폭에 가까운 크기(인치)
for temp, g in summary.groupby("temp_c"):
    ax.errorbar(
        g["cycle"], g["mean"], yerr=g["sd"],   # 오차막대 = 표준편차(캡션에 꼭 적습니다)
        capsize=2, linewidth=1, markersize=4,
        label=f"{temp} °C (n = {int(g['n'].iloc[0])})",
        **style[temp],
    )

ax.set_xlabel("Cycle number")
ax.set_ylabel("Capacity retention (%)")
ax.legend(frameon=False, fontsize=7)
ax.tick_params(labelsize=7)
ax.spines[["top", "right"]].set_visible(False)  # 불필요한 테두리 제거
fig.tight_layout()

# 벡터(PDF)와 고해상도 PNG 를 함께 저장합니다. 최종 형식은 학술지 지침을 따릅니다.
fig.savefig("fig1_retention.pdf")
fig.savefig("fig1_retention.png", dpi=300)
plt.show()